# Lab 9: Neural Networks

The PyTorch package provides an easy way to construct neural networks and handles all the optimization for us. Recall that we have already used PyTorch to optimize functions (see {ref}`S-pytorch`).

## Constructing a simple neural network

The following code builds a neural network model with 2 inputs, 3 hidden nodes (with ReLU activation), and 1 output (without activation). The neural network is implemented as an object derived from the *nn.Module* class (see <a href="https://lectures.scientific-python.org/intro/language/oop.html" target="_blank">the Python tutorial</a> if you are less familiar with class inheritance).

In [ ]:
import torch
import torch.nn as nn

class SimpleNN(nn.Module):
    def __init__(self):
        super(SimpleNN, self).__init__()
        self.fc1 = nn.Linear(2, 3) # 2 inputs, 3 hidden units
        self.relu = nn.ReLU() # ReLU activation
        self.fc2 = nn.Linear(3, 1) # 3 hidden units, 1 output

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

model = SimpleNN() # Creates the object

The different layers of the neural network are constructed in the "`__init__`" method of the class that runs automatically when an instance of the class is created. The "forward" method implements the actual calculation performed by the neural network. 

When the layers are created, their weights are automatically initialized. For example, let use print the weights and bias term of the first layer:

In [8]:
print(model.fc1.weight)
print(model.fc1.bias)

Parameter containing:
tensor([[-0.4291,  0.4496],
        [-0.5345, -0.4908],
        [-0.4533, -0.6010]], requires_grad=True)
Parameter containing:
tensor([-0.3641, -0.3326, -0.5572], requires_grad=True)


The neural network is nothing but a function from a particular parametric family of functions. Let us evaluate it at the point $(1,2)$:

In [14]:
x = torch.tensor([1,2], dtype=torch.float)
y = model(x)

print(y.item())

0.10422644764184952


## A neural network for the Iris dataset

Let us now train a neural network on the Iris dataset the we have considered several times already.

We begin by importing the necessary packages, loading the data, scaling it, encoding the categorical output feature, and splitting the data into a training and testing set.

In [17]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder

# Set random seed for reproducibility
torch.manual_seed(0)

# 1. Load the Iris dataset
iris = load_iris()
X = iris.data
y = iris.target

# 2. Preprocess the data
# Standardizing the features
scaler = StandardScaler()
X = scaler.fit_transform(X)

# One-hot encode the labels
encoder = OneHotEncoder(sparse_output=False)
y = encoder.fit_transform(y.reshape(-1, 1))

# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

# Convert numpy arrays to PyTorch tensors
X_train_tensor = torch.FloatTensor(X_train)
y_train_tensor = torch.FloatTensor(y_train)
X_test_tensor = torch.FloatTensor(X_test)
y_test_tensor = torch.FloatTensor(y_test)

Next, let us define a simple neural network to predict the labels. Recall that each flowers has 4 features (inputs) and that there are three species of Irises. As a result, our neural network needs 4 inputs, and 3 outputs to predict the one-encoded type of flower. Let us use one hidden layer with 10 neurons. That number (and the number of layers) can be changed later if the neural network is not good enough at predicting the species.

In [18]:
# 3. Define the neural network model
class SimpleNN(nn.Module):
    def __init__(self):
        super(SimpleNN, self).__init__()
        self.fc1 = nn.Linear(4, 10)  # 4 input features, 10 hidden units
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(10, 3)   # 10 hidden units, 3 output classes

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

model = SimpleNN()

Next, we need to tell PyTorch what loss function and what optimizer we want to use. Since we are working with categorical data, we will use the binary cross-entropy loss. The <a href="https://docs.pytorch.org/docs/stable/generated/torch.nn.BCEWithLogitsLoss.html" target="_blank">BCEWithLogitsLoss</a> method applies a sigmoid to the input and then evaluates the cross-entropy loss. Notice that we did not include any activation in the last layer of the network. Using this function instead is more numerically stable than using a plain Sigmoid followed by a BCELoss. 

For the optimizer, let us use the Adam optimizer with a learning rate of $0.01$. The learning rate can be changed as needed. A learning rate that is too large results in large steps in the gradient descent that prevent convergence. On the other hand, a very small learning rate makes the optimization very slow.  

In [19]:
# 4. Define loss function and optimizer
criterion = nn.BCEWithLogitsLoss()  # Binary Cross Entropy with logits
optimizer = optim.Adam(model.parameters(), lr=0.01)

Let us now run 1000 iterations of the Adam optimizer. In each iteration: 

1. We reset the gradient to zero.
2. We evaluate the function using the current model.
3. We evaluate the loss function.
4. We evaluate the gradient of the loss with respect to the different weights (loss.backward).
5. We perform a update of the parameters according to the Adam algorithm (optimizer.step).

Let us also print the loss at every 100 iterations.

In [22]:
# 5. Training loop
num_epochs = 1000

for epoch in range(num_epochs):
    model.train()  # Set the model to training mode
    optimizer.zero_grad()  # Zero the gradients
    outputs = model(X_train_tensor)  # Forward pass
    loss = criterion(outputs, y_train_tensor)  # Compute loss
    loss.backward()  # Backward pass
    optimizer.step()  # Update parameters

    if (epoch + 1) % 100 == 0:  # Print loss every 100 epochs
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')

Epoch [100/1000], Loss: 0.0251
Epoch [200/1000], Loss: 0.0246
Epoch [300/1000], Loss: 0.0242
Epoch [400/1000], Loss: 0.0239
Epoch [500/1000], Loss: 0.0237
Epoch [600/1000], Loss: 0.0235
Epoch [700/1000], Loss: 0.0234
Epoch [800/1000], Loss: 0.0233
Epoch [900/1000], Loss: 0.0232
Epoch [1000/1000], Loss: 0.0231


We are getting a pretty small training error so it looks like our neural network is fitting the training data well. 

Finally, let us evaluate the model on the testing set. This time, we are not using the BCEWithLogitsLoss function so we need to apply the sigmoid to the prediction and round it to get actual class labels. We can then measure the accuracy of our model on the testing set. 

In [21]:
# 6. Evaluate the model
model.eval()  # Set the model to evaluation mode
with torch.no_grad():
    test_outputs = model(X_test_tensor)
    predicted = torch.sigmoid(test_outputs).round()  # Apply sigmoid and round to get class labels
    accuracy = (predicted == y_test_tensor).float().mean().item() * 100
    print(f'Accuracy of the model on the test set: {accuracy:.2f}%')

Accuracy of the model on the test set: 100.00%


We obtain a perfect testing accuracy! 

```{note}

Let us take a moment to appreciate how powerful PyTorch is. It would take **a lot** of work to code the whole process we just implemented. Moreover, PyTorch can perform the same optimization for much larger neural network and datasets for us. With some small modifications of the code, we can even run it on a GPU to handle large models.  
```